# Loan Default Prediction – EDA, Modeling & Explainability

This notebook demonstrates the full pipeline:
1. Load data (real Lending Club sample or synthetic fallback)
2. Exploratory data analysis
3. Train / validation / test modelling with leakage-free protocol
4. SHAP explanations

**Important**: Run `python -m src.models.train` from the project root first if you want the saved pipeline artefact.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.generate_data import generate_loan_dataset
from src.data.load_data import load_raw_data
from src.features.preprocessing import prepare_xy, detect_schema, TARGET
from src.models.train import train_all_models, load_best_model

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)
%matplotlib inline

## 1. Load Data

In [ ]:
df = load_raw_data()
schema = detect_schema(df)
print(f"Schema: {schema} | Shape: {df.shape} | Default rate: {df[TARGET].mean():.3f}")
df.head()

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df[TARGET].value_counts().plot(kind="bar", ax=axes[0], color=["#2ecc71", "#e74c3c"])
axes[0].set_title("Target distribution")
axes[0].set_xlabel("default")

# Numeric correlation with target (only columns that exist)
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if TARGET in num_cols:
    corr = df[num_cols].corr()[TARGET].drop(TARGET).sort_values()
    corr.plot(kind="barh", ax=axes[1], color="steelblue")
    axes[1].set_title("Correlation with default")

plt.tight_layout()
plt.show()

In [ ]:
# Distribution of key numeric features by default status
key = [c for c in ["dti", "revol_util", "annual_inc", "borrower_score", "fico_score", "int_rate"] if c in df.columns]
if key:
    fig, axes = plt.subplots(1, len(key), figsize=(4 * len(key), 4))
    if len(key) == 1:
        axes = [axes]
    for ax, col in zip(axes, key):
        sns.boxplot(data=df, x=TARGET, y=col, ax=ax)
        ax.set_title(col)
    plt.tight_layout()
    plt.show()
else:
    print("No overlapping key numeric columns for boxplots.")

## 3. Train Models (Train / Val / Test protocol)

Model selection and threshold tuning use the **validation** set only.  
The **test** set is evaluated once after the best model is chosen.

In [ ]:
results = train_all_models()
print("\nValidation results:")
display(pd.DataFrame(results["val_results"]).T.round(4))
print("\nFinal test metrics:")
display(pd.Series(results["test_metrics"]).round(4))

## 4. SHAP Explanations

In [ ]:
from src.explainability.shap_analysis import compute_shap_values, plot_shap_summary
from src.data.load_data import load_processed_data

artifact = load_best_model()
pipe = artifact["pipeline"]

test_df = load_processed_data("test.parquet")
X_test, y_test = prepare_xy(test_df)

explainer, shap_vals, X_trans, feat_names = compute_shap_values(pipe, X_test, max_samples=300)
plot_shap_summary(shap_vals, X_trans, feat_names)
print("Top features by mean |SHAP|:")
mean_abs = np.abs(shap_vals).mean(axis=0)
order = np.argsort(mean_abs)[::-1][:10]
for i in order:
    print(f"  {feat_names[i]:40s} {mean_abs[i]:.4f}")